# WX-ES · Experiment 3 · Single-band GCN with matched-band PSD nodes (5 bands x PLV / |ciPLV|) · 200-epoch early stopping

The matched wMNE experiment `exp3_single_band` with the **early-stopping protocol**
`gnn_fixed_hparams_es200_v1` (methodology log entry 56, `docs/early_stopping.md`). It is identical to the 60-epoch
experiment except training:
- at most 200 epochs;
- patience 10, monitoring the inner-validation loss with min_delta 1e-4;
- the best epoch restored;
- no refit.

Hyperparameters, seeds, splits and reporting are unchanged, and the protocol is identical for LCMV and wMNE.

**No leakage:** in every outer fit, the inner validation participants (20 %, stratified, whole participants) are
drawn from that fit's training participants only, so the outer CV fold and the 13 holdout participants are never
used for stopping or checkpoint selection. - **10 variant(s):** `plv__delta`, `plv__theta`, `plv__alpha`, `plv__beta`, `plv__gamma`, `ciplv__delta`, `ciplv__theta`, `ciplv__alpha`, `ciplv__beta`, `ciplv__gamma`
- **Settings:** xu_dev_cv, xu_holdout (cross-dataset only if the DESIGN.md §7 gate passes)
- **Protocol (from the config, unchanged):** 1 repeats × 5-fold subject-level outer CV; fixed hyperparameters, no hyperparameter search and no inner CV: one fit per outer training fold (protocol `gnn_fixed_hparams_es200_v1`); 200 epochs.

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

**Suite:** matched wMNE counterpart of the D3 LCMV / Xu suite (methodology log entry 55, `docs/wmne_xu.md`). All
outputs are under `<outputs>/D3_WMNE_XU/`, with its own code pin `D3_WMNE_XU/CODE_COMMIT.txt`. The LCMV suite
(`D3_LCMV_XU/`: stores, results, pins), `graphsD3_v4_*` and D1/D2 are only read, never written. Safe to run while
LCMV experiments run in another Colab session (separate folders and pin; one GPU per session).
**Results** (separate from the 60-epoch results, never overwritten):
`<outputs>/D3_WMNE_XU/experiments_A_xu_tenfold/wx10_es_exp3_single_band/` and
`.../experiments_B_all_windows/wx_es_exp3_single_band/`. Every fold's `hparams.csv` row records the best / stopping epoch,
the best validation loss, the inner validation participants and both loss curves. Use a **GPU runtime** (T4).

In [ ]:
# ===== D3 wMNE / Xu suite setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_WX_CODE = False   # leave False. True only if you were asked to switch this suite to newer code
# The wMNE / Xu suite has its own code pin, <outputs>/D3_WMNE_XU/CODE_COMMIT.txt (all WX notebooks run
# that commit); the LCMV suite pin D3_LCMV_XU/CODE_COMMIT.txt and all other pins are only reported.
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--family', 'WX']
     + (['--update-code'] if UPDATE_WX_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)
print('Dataset 3 folder:', PATHS['dataset3']['root'])
LX = OUT + '/D3_LCMV_XU'
WX = OUT + '/D3_WMNE_XU'
print('suite outputs:', WX, '| LCMV reference (read only):', LX)

## 0. Code pin of the early-stopping protocol
The fixed 60-epoch results of this suite stay on the suite pin (unchanged; completed work is never invalidated).
This notebook runs the commit recorded in `D3_WMNE_XU/ES200_CODE_COMMIT.txt`. On its first
run it adopts the other suite's early-stopping commit if that exists, so LCMV and wMNE train with identical code;
otherwise it records the branch head. It never moves while early-stopping work of this suite exists. Leave
`UPDATE_ES_CODE = False`.

In [ ]:
UPDATE_ES_CODE = False
LX, WX = OUT + '/D3_LCMV_XU', OUT + '/D3_WMNE_XU'      # both suites' folders (the other one is only read)
import subprocess, sys
from pathlib import Path
from adgraph.colab import resolve_code_pin

head = subprocess.check_output(['git', '-C', '/content/repo', 'rev-parse', f'origin/{BRANCH}'], text=True).strip()
own_pin, other_pin = Path(WX) / 'ES200_CODE_COMMIT.txt', Path(LX) / 'ES200_CODE_COMMIT.txt'
sibling = other_pin.read_text().split()[0] if other_pin.exists() and other_pin.read_text().split() else None
target = head if UPDATE_ES_CODE else (sibling or head)
dirs = [Path(WX) / 'experiments_A_xu_tenfold', Path(WX) / 'experiments_B_all_windows']
has_work = any(any(d.glob(pre + '*/**/*.done.json')) for d in dirs if d.exists() for pre in ['wx_es_', 'wx10_es_'])
ES_COMMIT, _ = resolve_code_pin(own_pin, target, update=UPDATE_ES_CODE, has_work=has_work)
_run(['git', '-C', '/content/repo', 'checkout', '-q', ES_COMMIT])
if not Path('/content/repo/tests/test_early_stopping.py').exists():
    raise RuntimeError(f'pin {ES_COMMIT[:10]} lacks the early-stopping code: set UPDATE_ES_CODE = True and run again')
for m in [m for m in sys.modules if m == 'adgraph' or m.startswith('adgraph.')]:
    del sys.modules[m]                     # this notebook's own imports now come from the early-stopping commit
from adgraph.colab import sh
print('early-stopping code commit:', ES_COMMIT)
print('other suite early-stopping commit:', sibling or '(not yet set)')
if sibling and sibling != ES_COMMIT:
    print('WARNING: LCMV and wMNE early-stopping pins differ; the comparison would mix training code versions')
print('suite pin (60-epoch results, unchanged):', (Path(WX) / 'CODE_COMMIT.txt').read_text().split()[0])

## 1. Status (from the checkpoints on Drive)

In [ ]:
CONFIGS = {'A': 'configs/experiments/wx_es/tenfold/wx10_es_exp3_single_band.yaml', 'B': 'configs/experiments/wx_es/wx_es_exp3_single_band.yaml'}
import torch
print('runtime:', ('GPU ' + torch.cuda.get_device_name(0)) if torch.cuda.is_available() else 'CPU only')
for mode, cfg in CONFIGS.items():
    print('==== mode', mode)
    sh(f"python scripts/experiment.py status --config {cfg}")

## 2. Run mode A, then mode B (resumable)

In [ ]:
ERRORS = {}
for mode, cfg in CONFIGS.items():
    try:
        sh(f"python scripts/experiment.py run --config {cfg}")
    except RuntimeError as exc:
        ERRORS[mode] = exc
        print(f'mode {mode}: some units failed; details after the results')

## 3. Results and early-stopping summary (saved outputs only)

In [ ]:
import glob
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 300, 'display.width', 250)
for mode, cfg in CONFIGS.items():
    sh(f"python scripts/experiment.py report --config {cfg}")
    root = 'experiments_A_xu_tenfold' if mode == 'A' else 'experiments_B_all_windows'
    reports = sorted(glob.glob(f'{WX}/{root}/{Path(cfg).stem}/_reports/report_*'))
    if reports:
        print('==== mode', mode, reports[-1])
        display(pd.read_csv(reports[-1] + '/results_summary.csv'))
sh("python scripts/es_report.py --suite WX")
rep = sorted(glob.glob(WX + '/reports/early_stopping_*'))[-1]
display(pd.read_csv(rep + '/summary.csv').query("experiment.str.contains('exp3_single_band')", engine='python'))
for png in sorted(glob.glob(rep + '/curves_*exp3_single_band*.png')):
    display(Image(filename=png))
if ERRORS:
    raise RuntimeError(f'units did not complete in mode(s) {sorted(ERRORS)}; completed units are saved. Run all again.')